# Sentence Embeddings — AI Engineering Interview Prep

sentence-transformers, cosine similarity, semantic search, clustering.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Load a small, fast model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded: all-MiniLM-L6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 1. Basic Embeddings & Cosine Similarity

In [ ]:
sentences = [
    "Machine learning is a type of artificial intelligence.",
    "AI systems can learn from data automatically.",
    "Deep learning uses neural networks with many layers.",
    "The stock market closed higher today.",
    "Weather forecast: sunny skies expected this weekend.",
    "Neural networks are inspired by the human brain.",
]

embeddings = model.encode(sentences, normalize_embeddings=True)
print(f"Embedding shape: {embeddings.shape}  (n_sentences x dim)")

# Similarity matrix
sim_matrix = cosine_similarity(embeddings)

plt.figure(figsize=(8, 6))
plt.imshow(sim_matrix, cmap='coolwarm', vmin=0, vmax=1)
plt.colorbar()
labels = [s[:30] + '...' for s in sentences]
plt.xticks(range(len(sentences)), labels, rotation=45, ha='right', fontsize=8)
plt.yticks(range(len(sentences)), labels, fontsize=8)
plt.title("Cosine Similarity Matrix")
plt.tight_layout()
plt.show()

# Show most similar pairs
import itertools
pairs = [(i, j, sim_matrix[i,j]) for i,j in itertools.combinations(range(len(sentences)), 2)]
pairs.sort(key=lambda x: -x[2])
print("\nTop 3 similar pairs:")
for i, j, sim in pairs[:3]:
    print(f"  {sim:.4f}: '{sentences[i][:40]}...'")
    print(f"           '{sentences[j][:40]}...'")

## 2. Semantic Search

In [ ]:
corpus = [
    "How to implement gradient descent in Python?",
    "What is the difference between L1 and L2 regularization?",
    "Explain backpropagation in neural networks.",
    "How does attention mechanism work in transformers?",
    "What are the advantages of random forest over decision trees?",
    "How to handle class imbalance in machine learning?",
    "What is the vanishing gradient problem?",
    "How to evaluate a language model's quality?",
    "Explain k-fold cross validation.",
    "What is the difference between precision and recall?"
]

# Encode corpus (in production: cache these!)
corpus_embeddings = model.encode(corpus, normalize_embeddings=True)

def semantic_search(query, corpus, corpus_embeddings, top_k=3):
    q_emb = model.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(q_emb, corpus_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(corpus[i], scores[i]) for i in top_indices]

queries = [
    "explain how neural nets learn",
    "dealing with unbalanced datasets",
]

for q in queries:
    print(f"\nQuery: '{q}'")
    results = semantic_search(q, corpus, corpus_embeddings)
    for doc, score in results:
        print(f"  [{score:.4f}] {doc}")

## 3. Embedding Visualization with t-SNE

In [ ]:
from sklearn.manifold import TSNE

# Category-labeled sentences
ml_sentences = [
    "Neural networks learn by adjusting weights.",
    "Gradient descent minimizes the loss function.",
    "Backpropagation computes gradients efficiently.",
    "Convolutional networks excel at image tasks.",
    "Transformers use self-attention mechanisms.",
]
data_sentences = [
    "Pandas is used for data manipulation.",
    "SQL queries retrieve structured data.",
    "Feature engineering improves model performance.",
    "Missing values can be imputed with the mean.",
    "Normalization scales features to [0,1].",
]
eval_sentences = [
    "F1 score balances precision and recall.",
    "Cross-validation prevents overfitting.",
    "AUC-ROC measures classification performance.",
    "Confusion matrix shows prediction errors.",
    "Learning curves diagnose bias and variance.",
]

all_sents   = ml_sentences + data_sentences + eval_sentences
all_labels  = ["Deep Learning"]*5 + ["Data Science"]*5 + ["Evaluation"]*5
all_embeddings = model.encode(all_sents, normalize_embeddings=True)

tsne = TSNE(n_components=2, random_state=42, perplexity=5)
reduced = tsne.fit_transform(all_embeddings)

colors = {'Deep Learning': 'blue', 'Data Science': 'green', 'Evaluation': 'red'}
plt.figure(figsize=(8, 6))
for label in set(all_labels):
    mask = [l == label for l in all_labels]
    pts = reduced[mask]
    plt.scatter(pts[:,0], pts[:,1], c=colors[label], label=label, s=100, alpha=0.8)
for i, sent in enumerate(all_sents):
    plt.annotate(sent[:20], reduced[i], fontsize=7, alpha=0.7)
plt.legend()
plt.title("Sentence Embeddings t-SNE")
plt.tight_layout()
plt.show()

## Interview Tips

- **Cosine similarity vs dot product**: with normalized embeddings they're equivalent. Always normalize.
- **Semantic search pipeline**: embed corpus offline → store in vector DB → embed query at runtime → ANN search.
- **Embedding models**: `all-MiniLM-L6-v2` (fast, small); `text-embedding-3-large` (OpenAI, high quality); `voyage-3` (Anthropic).
- **Retrieval evaluation**: MRR@k (Mean Reciprocal Rank), Recall@k, NDCG@k.
- **Embedding dimension tradeoff**: larger dim = richer representation but more storage/compute.
- **Fine-tuning embeddings**: use contrastive learning (Sentence-BERT style) with positive/negative pairs.